In [1]:
import os
from collections import defaultdict
from itertools import chain, product
from pathlib import Path
from social_groups.config import GROUP_SIZES, MODEL_LETTERS

In [2]:
model_families = {
    "Qwen3": {
        "L": "qwen-0_6b",
        "M": "qwen-4b",
        "H": "qwen-14b",
    },
    "Qwen35": {
        "L": "qwen35-0_8b",
        "M": "qwen35-4b",
        "H": "qwen35-9b",
    },
    "Ministral3": {
        "L": "ministral3-3b",
        "M": "ministral3-8b",
        "H": "ministral3-14b",
    },
}

GROUPINGS = list(
    chain(
        *[["".join(c) for c in product(MODEL_LETTERS, repeat=i)] for i in GROUP_SIZES]
    )
)

In [3]:
for family, models in model_families.items():
    for grouping in GROUPINGS:
        file = Path(family, grouping + ".yaml")

        os.makedirs(file.parent, exist_ok=True)

        if os.path.exists(file):
            os.remove(file)

        with open(file, "x") as f:
            f.writelines(["# @package _global_\n", "defaults:\n"])
            f.writelines(
                [
                    f"  - /experiment/hetero/backend@{i}: {models[m]}\n"
                    for i, m in enumerate(grouping)
                ]
            )

In [27]:
family_prefix_mapping = {
    "Qwen3": "q",
    "Qwen35": "Q",
    "Ministral3": "m",
}

mixed_model_names = {
    f"{letter}{family_prefix_mapping[family]}": label
    for family, models in model_families.items()
    for letter, label in models.items()
}


def same_family(g):
    return all(g[0][1] == x[1]  for x in g)

def all_different_families(g):
    return len(g) == len(set(x[1] for x in g))

def all_same_relative_size(g):
    return all(g[0][0] == x[0]  for x in g)

MIXED_GROUPINGS = list(
    filter(
        lambda x: not same_family(x),
        chain(
            *[
                [c for c in product(mixed_model_names.keys(), repeat=i)]
                for i in [2,3]
            ]
        ),
    )
)

for g in MIXED_GROUPINGS:
    grouping = "".join(g)

    file = Path("mixed", grouping + ".yaml")

    os.makedirs(file.parent, exist_ok=True)

    if os.path.exists(file):
        os.remove(file)

    with open(file, "x") as f:
        f.writelines(["# @package _global_\n", "defaults:\n"])
        f.writelines(
            [
                f"  - /experiment/hetero/backend@{i}: {mixed_model_names[m]}\n"
                for i, m in enumerate(g)
            ]
        )

## TO COPY:

## Only one each:

In [4]:
sizes = defaultdict(set)


for family in model_families:
    for grouping in GROUPINGS:
        sizes[len(grouping)].add("".join(sorted(grouping)))

    for group in sizes.values():
        print(f"experiment/hetero/groups/{family}@_models: ", ", ".join(group))

experiment/hetero/groups/Qwen3@_models:  H, M, L
experiment/hetero/groups/Qwen3@_models:  HM, HH, MM, LL, HL, LM
experiment/hetero/groups/Qwen3@_models:  HLL, HHH, HLM, LMM, HMM, HHM, MMM, LLM, HHL, LLL
experiment/hetero/groups/Qwen3@_models:  HHHM, HLMM, HHHH, HHHL, LLLL, LMMM, HHLM, HLLM, HMMM, MMMM, HHMM, HLLL, HHLL, LLLM, LLMM
experiment/hetero/groups/Qwen35@_models:  H, M, L
experiment/hetero/groups/Qwen35@_models:  HM, HH, MM, LL, HL, LM
experiment/hetero/groups/Qwen35@_models:  HLL, HHH, HLM, LMM, HMM, HHM, MMM, LLM, HHL, LLL
experiment/hetero/groups/Qwen35@_models:  HHHM, HLMM, HHHH, HHHL, LLLL, LMMM, HHLM, HLLM, HMMM, MMMM, HHMM, HLLL, HHLL, LLLM, LLMM
experiment/hetero/groups/Ministral3@_models:  H, M, L
experiment/hetero/groups/Ministral3@_models:  HM, HH, MM, LL, HL, LM
experiment/hetero/groups/Ministral3@_models:  HLL, HHH, HLM, LMM, HMM, HHM, MMM, LLM, HHL, LLL
experiment/hetero/groups/Ministral3@_models:  HHHM, HLMM, HHHH, HHHL, LLLL, LMMM, HHLM, HLLM, HMMM, MMMM, HHMM, 

# All Permutations:

In [5]:
sizes = defaultdict(list)


for family in model_families:
    for grouping in GROUPINGS:
        sizes[len(grouping)].append("".join(grouping))

    for group in sizes.values():
        print(f"experiment/hetero/groups/{family}@_models: ", ", ".join(group))

experiment/hetero/groups/Qwen3@_models:  L, M, H
experiment/hetero/groups/Qwen3@_models:  LL, LM, LH, ML, MM, MH, HL, HM, HH
experiment/hetero/groups/Qwen3@_models:  LLL, LLM, LLH, LML, LMM, LMH, LHL, LHM, LHH, MLL, MLM, MLH, MML, MMM, MMH, MHL, MHM, MHH, HLL, HLM, HLH, HML, HMM, HMH, HHL, HHM, HHH
experiment/hetero/groups/Qwen3@_models:  LLLL, LLLM, LLLH, LLML, LLMM, LLMH, LLHL, LLHM, LLHH, LMLL, LMLM, LMLH, LMML, LMMM, LMMH, LMHL, LMHM, LMHH, LHLL, LHLM, LHLH, LHML, LHMM, LHMH, LHHL, LHHM, LHHH, MLLL, MLLM, MLLH, MLML, MLMM, MLMH, MLHL, MLHM, MLHH, MMLL, MMLM, MMLH, MMML, MMMM, MMMH, MMHL, MMHM, MMHH, MHLL, MHLM, MHLH, MHML, MHMM, MHMH, MHHL, MHHM, MHHH, HLLL, HLLM, HLLH, HLML, HLMM, HLMH, HLHL, HLHM, HLHH, HMLL, HMLM, HMLH, HMML, HMMM, HMMH, HMHL, HMHM, HMHH, HHLL, HHLM, HHLH, HHML, HHMM, HHMH, HHHL, HHHM, HHHH
experiment/hetero/groups/Qwen35@_models:  L, M, H, L, M, H
experiment/hetero/groups/Qwen35@_models:  LL, LM, LH, ML, MM, MH, HL, HM, HH, LL, LM, LH, ML, MM, MH, HL, HM, HH
ex

### Mixed Groupings

In [35]:
sizes = defaultdict(list)

print("All Different ones: ")
for g in MIXED_GROUPINGS:
    if not all_different_families(g):
        continue
    if not all_same_relative_size(g):
        continue
    grouping = "".join(g)

    s = "".join(x for i, x in enumerate(grouping) if (i % 2) == 0 )

    sizes[s].append(grouping)



for group in sizes.values():
    print(f"experiment/hetero/groups/mixed@_models: ", ", ".join(group))

All Different ones: 
experiment/hetero/groups/mixed@_models:  LqLQ, LqLm, LQLq, LQLm, LmLq, LmLQ
experiment/hetero/groups/mixed@_models:  MqMQ, MqMm, MQMq, MQMm, MmMq, MmMQ
experiment/hetero/groups/mixed@_models:  HqHQ, HqHm, HQHq, HQHm, HmHq, HmHQ
experiment/hetero/groups/mixed@_models:  LqLQLm, LqLmLQ, LQLqLm, LQLmLq, LmLqLQ, LmLQLq
experiment/hetero/groups/mixed@_models:  MqMQMm, MqMmMQ, MQMqMm, MQMmMq, MmMqMQ, MmMQMq
experiment/hetero/groups/mixed@_models:  HqHQHm, HqHmHQ, HQHqHm, HQHmHq, HmHqHQ, HmHQHq
